<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/04-series-de-tiempo/ejercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Ejercicio 2.4 · Series de tiempo: día de la semana y feriados

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Antes de empezar, guarda tu copia: **Archivo → Guardar una copia en Drive**.

Escribe tu respuesta en cada celda de código y ejecútala con **Shift + Enter**.

### Prepara los datos

Ejecuta estas celdas. Dejan listas `serie`, `FERIADOS` y `tres_pronosticos`. `serie` debe tener `730` días.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [ ]:
FERIADOS = pd.to_datetime(
    [
        "2024-01-01",
        "2024-03-28",
        "2024-03-29",
        "2024-05-01",
        "2024-06-29",
        "2024-07-28",
        "2024-07-29",
        "2024-08-30",
        "2024-10-08",
        "2024-11-01",
        "2024-12-08",
        "2024-12-09",
        "2024-12-25",
        "2025-01-01",
        "2025-04-17",
        "2025-04-18",
        "2025-05-01",
        "2025-06-29",
        "2025-07-28",
        "2025-07-29",
        "2025-08-30",
        "2025-10-08",
        "2025-11-01",
        "2025-12-08",
        "2025-12-09",
        "2025-12-25",
    ]
)

In [ ]:
def calendario(dias):
    """Una columna por día de la semana y por mes, más la tendencia: los días desde el inicio."""
    x = pd.DataFrame(index=dias)
    for dia in range(7):
        x[f"dia_{dia}"] = (dias.dayofweek == dia).astype(int)
    for mes in range(1, 13):
        x[f"mes_{mes}"] = (dias.month == mes).astype(int)
    x["tendencia"] = (dias - pd.Timestamp("2024-01-01")).days
    return x

In [ ]:
def tres_pronosticos(serie, feriados, mes):
    """Pronostica cada día de un mes con tres métodos, usando solo lo anterior a ese mes."""
    inicio = pd.Timestamp(mes + "-01")
    dias = pd.date_range(inicio, inicio + pd.offsets.MonthEnd(0))
    antes = serie[serie.index < inicio]
    laborables = antes[~antes.index.isin(feriados)]

    # Promedio por día de la semana de las últimas 8 semanas.
    ultimas = laborables[laborables.index >= inicio - pd.Timedelta(days=56)]
    por_dia = ultimas.groupby(ultimas.index.dayofweek).mean()
    ocho_semanas = pd.Series(dias.dayofweek.map(por_dia), index=dias)

    # Índice: cuánto subió ese mes en 2024 frente a las 8 semanas previas de 2024.
    hace_un_anio = inicio - pd.Timedelta(days=364)
    base = laborables[
        (laborables.index >= hace_un_anio - pd.Timedelta(days=56))
        & (laborables.index < hace_un_anio)
    ]
    mismo_mes = laborables[
        (laborables.index.year == 2024) & (laborables.index.month == inicio.month)
    ]
    regla = ocho_semanas * mismo_mes.mean() / base.mean()

    modelo = LinearRegression().fit(calendario(laborables.index), laborables)
    regresion = pd.Series(modelo.predict(calendario(dias)), index=dias)

    # Todos los métodos ponen los feriados en cero: esos días no se vende.
    tabla = pd.DataFrame(
        {"real": serie[dias], "ocho_semanas": ocho_semanas, "regla": regla, "regresion": regresion}
    )
    metodos = ["ocho_semanas", "regla", "regresion"]
    tabla.loc[tabla.index.isin(feriados), metodos] = 0
    return tabla

In [ ]:
ventas = pd.read_csv(DATOS + "ventas.csv", parse_dates=["fecha"])

serie = ventas.groupby("fecha")["total"].sum().asfreq("D", fill_value=0)

print(len(serie))

### 1. El mejor día

¿Qué día de la semana tiene la mayor venta promedio? ¿Y el menor? Responde con `serie.groupby(serie.index.dayofweek).mean()`, recordando que 0 es lunes.

In [ ]:
# Escribe tu código aquí

### 2. Los feriados

Cuenta cuántos días de la serie valen exactamente cero, y cuántos feriados hay en la lista. Deben ser `25` y `26`. ¿Por qué no coinciden? Mira el primer día de `serie`.

In [ ]:
# Escribe tu código aquí

### 3. Otro mes

Calcula `tres_pronosticos(serie, FERIADOS, "2025-09")` y guárdalo en `septiembre`. Calcula el error medio de cada método. La regla debe dar `2,485`.

In [ ]:
# Escribe tu código aquí

### 4. ¿Y por mes?

Suma la venta de `septiembre`, real y de cada método, con `.sum()`. ¿Qué método se acerca más al total del mes?

In [ ]:
# Escribe tu código aquí

### 5. Sin el índice

Compara el error de `ocho_semanas` y de `regla` en `septiembre`. Responde en un comentario: ¿ayudó el índice? ¿Podrías hacer esto en Excel?

In [ ]:
# Escribe tu código aquí

### 6. ¿Cómo lo presentarías?

Responde en un comentario. Tu jefe pregunta: *"¿Cuál método usamos para siempre?"* Con lo que viste en dos meses, ¿qué le contestas?

In [ ]:
# Escribe tu código aquí

## 🎉 ¡Terminaste!

Compara tus respuestas con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/04-series-de-tiempo/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0